In [0]:
%sql
USE CATALOG students_data;
USE SCHEMA `team3-taxi`;

In [0]:
%sql
CREATE OR REPLACE TABLE fact_trip AS
SELECT
    ------------------------------------------------------------------
    -- DIMENSION SURROGATE KEYS
    -- These replace natural keys (IDs, timestamps, coordinates)
    -- with stable, conformed surrogate keys from the dimension tables.
    ------------------------------------------------------------------
    dv.vendor_key,
    dr.rate_code_key,
    dp.payment_type_key,

    pl.location_key AS pickup_location_key,
    dl.location_key AS dropoff_location_key,

    ------------------------------------------------------------------
    -- TIME DIMENSIONS (MULTI-GRAIN)
    -- We join to day, hour, and minute grains so analysts can choose
    -- the appropriate level of temporal analysis.
    --
    -- Day grain: good for daily trends, seasonality, holidays.
    -- Hour grain: good for peak-hour analysis, traffic patterns.
    -- Minute grain: good for ML features, congestion modelling.
    ------------------------------------------------------------------

    -- Date grain (1 row per date)
    dday_p.datetime_day_key AS pickup_datetime_day_key,
    dday_d.datetime_day_key AS dropoff_datetime_day_key,

    -- Hour grain (1 row per hour)
    dhour_p.datetime_hour_key AS pickup_datetime_hour_key,
    dhour_d.datetime_hour_key AS dropoff_datetime_hour_key,

    -- Minute grain (1 row per minute)
    dmin_p.datetime_minute_key AS pickup_datetime_minute_key,
    dmin_d.datetime_minute_key AS dropoff_datetime_minute_key,

    ------------------------------------------------------------------
    -- BASE MEASURES (RAW FACTS)
    -- These come directly from the Silver layer with minimal logic.
    -- They represent the atomic facts of a single taxi trip.
    ------------------------------------------------------------------
    fs.trip_distance,              -- Distance travelled in miles
    fs.trip_duration_minutes,      -- Duration of trip in minutes
    fs.fare_amount,                -- Base fare charged
    fs.extra,                      -- Extra charges (e.g., rush hour)
    fs.mta_tax,                    -- MTA tax
    fs.tip_amount,                 -- Tip paid by passenger
    fs.tolls_amount,               -- Tolls charged
    fs.improvement_surcharge,      -- NYC improvement surcharge
    fs.total_amount,               -- Total amount charged to passenger

    ------------------------------------------------------------------
    -- ATTRIBUTES
    ------------------------------------------------------------------
    fs.passenger_count,            -- Number of passengers
    fs.store_and_fwd_flag,         -- Whether trip was stored & forwarded
    fs.trip_id,                    -- Technical surrogate key for the trip

    ------------------------------------------------------------------
    -- BUSINESS METRICS (DERIVED KPIs)
    -- These are the metrics analysts actually use.
    -- They encode business logic directly into the model so that
    -- dashboards and queries remain consistent across the company.
    ------------------------------------------------------------------

    -- Revenue per mile:
    -- Measures profitability of distance travelled.
    -- Useful for route efficiency and driver performance.
    CASE WHEN fs.trip_distance > 0 
         THEN fs.total_amount / fs.trip_distance 
    END AS revenue_per_mile,

    -- Tip percentage:
    -- Measures customer tipping behaviour relative to fare.
    -- Useful for payment type analysis and driver performance.
    CASE WHEN fs.fare_amount > 0 
         THEN fs.tip_amount / fs.fare_amount 
    END AS tip_pct,

    -- Fare per minute:
    -- Normalises fare by time, useful for congestion analysis.
    CASE WHEN fs.trip_duration_minutes > 0 
         THEN fs.fare_amount / fs.trip_duration_minutes 
    END AS fare_per_minute,

    -- Fare per passenger:
    -- Useful for analysing group rides and occupancy efficiency.
    CASE WHEN fs.passenger_count > 0 
         THEN fs.total_amount / fs.passenger_count 
    END AS fare_per_passenger,

    -- Total charges (excluding tip):
    -- Represents the operational revenue components.
    (fs.fare_amount 
     + fs.extra 
     + fs.mta_tax 
     + fs.tolls_amount 
     + fs.improvement_surcharge) AS total_charges,

    -- Effective rate:
    -- Total revenue per minute of trip time.
    -- Useful for comparing short vs long trips.
    CASE WHEN fs.trip_duration_minutes > 0 
         THEN fs.total_amount / fs.trip_duration_minutes 
    END AS effective_rate,

    -- Average speed in mph:
    -- Derived from distance and duration.
    -- Useful for traffic, congestion, and operational analysis.
    CASE WHEN fs.trip_duration_minutes > 0
         THEN fs.trip_distance / (fs.trip_duration_minutes / 60) 
    END AS avg_speed_mph

FROM fact_trip_silver fs

----------------------------------------------------------------------
-- DIMENSION JOINS
-- These map natural keys from Silver to surrogate keys in Gold.
----------------------------------------------------------------------

LEFT JOIN dim_vendor dv 
    ON fs.vendor_id = dv.vendor_id

LEFT JOIN dim_rate_code dr 
    ON fs.rate_code_id = dr.rate_code_id

LEFT JOIN dim_payment_type dp 
    ON fs.payment_type_id = dp.payment_type_id

LEFT JOIN dim_location pl 
    ON fs.pickup_latitude = pl.latitude
   AND fs.pickup_longitude = pl.longitude

LEFT JOIN dim_location dl 
    ON fs.dropoff_latitude = dl.latitude
   AND fs.dropoff_longitude = dl.longitude

----------------------------------------------------------------------
-- TIME DIMENSION JOINS
-- We join the same timestamps to three different grains.
----------------------------------------------------------------------

-- Day grain
LEFT JOIN dim_datetime_day dday_p 
    ON DATE(fs.pickup_datetime) = dday_p.date_value

LEFT JOIN dim_datetime_day dday_d 
    ON DATE(fs.dropoff_datetime) = dday_d.date_value

-- Hour grain
LEFT JOIN dim_datetime_hour dhour_p 
    ON DATE_TRUNC('hour', fs.pickup_datetime) = dhour_p.datetime_value

LEFT JOIN dim_datetime_hour dhour_d 
    ON DATE_TRUNC('hour', fs.dropoff_datetime) = dhour_d.datetime_value

-- Minute grain
LEFT JOIN dim_datetime_minute dmin_p 
    ON DATE_TRUNC('minute', fs.pickup_datetime) = dmin_p.datetime_value

LEFT JOIN dim_datetime_minute dmin_d 
    ON DATE_TRUNC('minute', fs.dropoff_datetime) = dmin_d.datetime_value;
